# Notebook 5: BERT Experiment
Fine-tunes BERT-base-uncased and evaluates under in-distribution and cross-generator conditions.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '..')
from src.features.transformer import build_dataset
from src.models.bert_model import train_bert, predict, load_model
from src.evaluation.metrics import evaluate, cross_gen_drop

train = pd.read_csv('../data/processed/train.csv')
val   = pd.read_csv('../data/processed/val.csv')
test  = pd.read_csv('../data/processed/test_in_dist.csv')
test_cg = pd.read_csv('../data/processed/test_cross_gen.csv')

MODEL = 'bert-base-uncased'

In [ ]:
train_ds = build_dataset(train['text'].tolist(), train['label'].tolist(), MODEL)
val_ds   = build_dataset(val['text'].tolist(),   val['label'].tolist(),   MODEL)
test_ds  = build_dataset(test['text'].tolist(),  test['label'].tolist(),  MODEL)
test_cg_ds = build_dataset(test_cg['text'].tolist(), test_cg['label'].tolist(), MODEL)

In [ ]:
model = train_bert(train_ds, val_ds, model_name=MODEL)

In [ ]:
probs = predict(model, test_ds)
result_id = evaluate(test['label'], (probs >= 0.5).astype(int), probs, label='BERT (in-dist)')

In [ ]:
probs_cg = predict(model, test_cg_ds)
result_cg = evaluate(test_cg['label'], (probs_cg >= 0.5).astype(int), probs_cg, label='BERT (cross-gen)')
cross_gen_drop(result_id['f1'], result_cg['f1'])